# 02 · Parcele cu Fields of The World (pe A100)
Descarcă două scene Sentinel-2 (primăvară, final de vară) de pe Planetary Computer pentru fiecare tile MGRS din zonă, rulează modelul FTW și unește poligoanele în `data/out/parcels.parquet` (EPSG:3844).

**Atenție:** zona traversează limita UTM 34/35 la 24° E, deci sunt cel puțin două tile-uri.

In [ ]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul gisday2026): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/gisday2026.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/gisday2026'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/gisday2026'], check=True)
        os.chdir('/content/gisday2026')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

In [ ]:
v = sys.version_info
print(f'Python {v.major}.{v.minor}')
FTW = 'ftw'
if not ((3, 11) <= (v.major, v.minor) < (3, 13)):
    print('ftw-tools cere Python >=3.11,<3.13 -> rulează celula următoare (mediu separat cu Python 3.12)')

In [ ]:
# DOAR dacă Python-ul runtime-ului nu e 3.11/3.12: mediu separat pentru FTW
# !pip install -q uv && uv venv -p 3.12 /content/ftwenv && /content/ftwenv/bin/python -m ensurepip
# !/content/ftwenv/bin/python -m pip install -q ftw-tools
# FTW = '/content/ftwenv/bin/ftw' 

In [ ]:
%pip install -q ftw-tools pystac-client planetary-computer geopandas pyarrow shapely

In [ ]:
# Verifică opțiunile reale ale CLI-ului înainte de rulare (nu ghicim flag-urile)
for sub in (['--help'], ['inference', 'download', '--help'], ['inference', 'run', '--help'],
            ['inference', 'polygonize', '--help'], ['model', 'download', '--help']):
    r = subprocess.run([FTW, *sub], capture_output=True, text=True)
    print('$ ftw', ' '.join(sub)); print(r.stdout[-1500:] or r.stderr[-1500:]); print('-' * 60)

In [ ]:
subprocess.run([FTW, 'model', 'download', '--type', 'THREE_CLASS_FULL'], check=True)
import glob
CKPT = sorted(glob.glob('**/3_Class_FULL_FTW_Pretrained.ckpt', recursive=True) + glob.glob('*.ckpt'))[0]
print('checkpoint:', CKPT)

In [ ]:
import pystac_client, planetary_computer
from shapely.geometry import box, shape
cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                modifier=planetary_computer.sign_inplace)
aoi_ll = box(*AOI_BBOX)

def best(window, max_cloud=10):
    items = cat.search(collections=['sentinel-2-l2a'], bbox=AOI_BBOX, datetime=window,
                       query={'eo:cloud_cover': {'lt': max_cloud}}).item_collection()
    pick = {}
    for it in items:
        t = it.properties.get('s2:mgrs_tile')
        score = (round(shape(it.geometry).intersection(aoi_ll).area, 4), -it.properties['eo:cloud_cover'])
        if t not in pick or score > pick[t][0]:
            pick[t] = (score, it)
    return {t: p[1] for t, p in pick.items()}

A = best(f'{YEAR}-04-01/{YEAR}-05-31')
B = best(f'{YEAR}-08-01/{YEAR}-09-30')
TILES = sorted(set(A) & set(B))
for t in TILES:
    print(t, A[t].id, A[t].properties['eo:cloud_cover'], '|', B[t].id, B[t].properties['eo:cloud_cover'])

In [ ]:
poly_help = subprocess.run([FTW, 'inference', 'polygonize', '--help'], capture_output=True, text=True).stdout
outputs = []
for t in TILES:
    a, b = A[t], B[t]
    inter = shape(a.geometry).intersection(shape(b.geometry)).intersection(aoi_ll)
    if inter.is_empty or inter.area < 1e-4:
        continue
    bb = ','.join(f'{c:.5f}' for c in inter.bounds)
    tif, pred = INTERIM / f'ftw_in_{t}.tif', INTERIM / f'ftw_pred_{t}.tif'
    subprocess.run([FTW, 'inference', 'download', '--win_a', a.id, '--win_b', b.id, '--out', str(tif), '--bbox', bb], check=True)
    subprocess.run([FTW, 'inference', 'run', str(tif), '--model', CKPT, '--out', str(pred), '--gpu', '0', '--overwrite'], check=True)
    before = set(INTERIM.glob('*'))
    cmd = [FTW, 'inference', 'polygonize', str(pred), '--simplify', '20']
    if '--out' in poly_help:
        cmd += ['--out', str(INTERIM / f'ftw_poly_{t}.parquet')]
    subprocess.run(cmd, check=True)
    new = [p for p in set(INTERIM.glob('*')) - before if p.suffix in ('.parquet', '.gpkg', '.geojson', '.shp')]
    outputs += new or [INTERIM / f'ftw_poly_{t}.parquet']
    print(t, '->', outputs[-1])

In [ ]:
import geopandas as gpd, pandas as pd
parts = []
for p in outputs:
    g = gpd.read_parquet(p) if p.suffix == '.parquet' else gpd.read_file(p)
    parts.append(g.to_crs(CRS)[['geometry']])
P = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=CRS)
P = P[P.geometry.is_valid & ~P.geometry.is_empty]
# duplicate la suprapunerea tile-urilor: păstrează poligonul mai mare când IoU > 0.5
P['a'] = P.area
drop = set()
for i, j in zip(*P.sindex.query(P.geometry, predicate='intersects')):
    if i < j and i not in drop and j not in drop:
        gi, gj = P.geometry.iloc[i], P.geometry.iloc[j]
        iou = gi.intersection(gj).area / gi.union(gj).area
        if iou > 0.5:
            drop.add(j if P.a.iloc[i] >= P.a.iloc[j] else i)
P = P.drop(P.index[list(drop)])

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

In [ ]:
from shapely.geometry import shape as _shape
aoi_poly = gpd.GeoSeries([_shape(AOI.getInfo())], crs='EPSG:4326').to_crs(CRS).iloc[0]
P = P[P.intersects(aoi_poly)].copy()
P['geometry'] = P.intersection(aoi_poly)
P['area_ha'] = P.area / 1e4
P = P[P.area_ha >= 0.5].reset_index(drop=True)
P['parcel_id'] = range(1, len(P) + 1)
P[['parcel_id', 'area_ha', 'geometry']].to_parquet(OUT / 'parcels.parquet')
print(len(P), 'parcele;', round(P.area_ha.sum()), 'ha')

In [ ]:
push([OUT / 'parcels.parquet'], 'parcele FTW')